# PlanetScope acquisition — Folger Deep vessel-noise study

Find PlanetScope scenes that **fully contain** the Folger core AOI with **minimal cloud over
that AOI**, review the whole AOI for vessels using the tile quota, then order only the ones
worth paying for.

The AOI is a 10 × 10 km box centred on the hydrophone at −125.278277, 48.814200, inset 5 m
per edge to **99.800 km²**. The inset is deliberate. The nominal box measures 100.00024 km²,
and since Planet's clip-area accounting is exact, thirty orders of it would ask for
3000.007 km² against a 3000 km² quota and the thirtieth would be refused. The inset also
covers a second, larger exposure: a lat/lon rectangle projects to a quadrilateral whose
*bounding box* exceeds its polygon area by ~90,000 m², and a clipped GeoTIFF is a rectangle —
so if Planet bills the output footprint rather than the cut polygon, the bounding box is what
counts. At 5 m both come in under the line (99.800 and 99.889 km²), each order is billed the
flat 100 km² clip minimum, and thirty come to exactly 3,000. Stage 1 derives all of this from
the geojson and asserts it, so the notebook cannot drift from the file on disk.

The cost of the inset is 5 m of extent per edge — under two pixels at 3 m — and 0.2% of area.

**Environment:** CryoCloud JupyterHub, ~4 GB RAM, Planet SDK v2.

## Two separate budgets

| Budget | Allocation | Cost per scene | Ceiling |
|---|---|---|---|
| **Imagery** | 3,000 km²/month | 100 km² minimum when clipping | **30 scenes/month** |
| **Scene tiles** | ~98,000 tiles/month | 196 tiles (full AOI) at z15 | ~500 previews |

Tiles, quicklooks and UDM2 masks are all free against the imagery budget, so everything up to
stage 9 costs nothing. Only stage 9 spends, behind a manual gate.

## Selection logic

Three hard gates, then your eyes:

1. **Coverage** — the scene footprint must *contain* the whole AOI, tested topologically in
   an equal-area projection, not by an area-ratio threshold
2. **Cloud** — measured on the AOI itself via free UDM2, not on the scene footprint
3. **Vessel present** — confirmed by reviewing tile previews *of the whole AOI*

All three are measured on the same footprint — the same one `clip_tool` bills against — so a
scene is never selected on one extent and judged on another. Whatever passes all three gets
ordered, cleanest first, up to the monthly cap. There is no weighted scoring function and no
stratification by year: the gates are the selection, and `aoi_clear` breaks ties.

## Order of operations

1. Setup and AOI
2. Search — deliberately loose
3. Coverage filter
4. UDM2 cloud screening on the AOI
5. Glint (review-queue ordering only)
6. Quicklook skim
7. Tile preview and vessel review
8. Acoustic join — **optional**
9. Select and order

## 1. Setup

In [ ]:
%pip install --quiet "planet>=2.1,<3" pandas rasterio pillow httpx shapely pyproj

In [ ]:
import asyncio, json, math, os
from datetime import datetime, timedelta
from pathlib import Path
from io import BytesIO

import numpy as np
import pandas as pd
import httpx
import rasterio
from PIL import Image
from shapely.geometry import shape
from shapely.ops import transform as _transform
from pyproj import Transformer
from rasterio.windows import from_bounds
from rasterio.features import geometry_mask, bounds as geom_bounds
from rasterio.warp import transform_geom
from planet import Auth, Session, data_filter, order_request, reporting

# ------------------------------------------------------------------ paths
AOI_PATH = Path("folger_core_aoi.geojson")
WORK     = Path("planet_folger");   WORK.mkdir(exist_ok=True)
UDM_DIR  = WORK / "udm2_scratch";   UDM_DIR.mkdir(exist_ok=True)
PREVIEW  = WORK / "tile_previews";  PREVIEW.mkdir(exist_ok=True)

# ------------------------------------------------------------------ search
YEARS     = range(2020, 2026)
SUMMER    = (6, 9)                 # June 1 -> Sept 1, exclusive upper bound
ITEM_TYPE = "PSScene"

# Loose on purpose. cloud_cover describes the whole ~637 km2 footprint; the AOI is
# ~15% of it. The free UDM2 screen in stage 4 measures cloud where it matters.
SEARCH_CLOUD_MAX = 0.60

# ------------------------------------------------------------------ the three gates
# 1. Coverage. Planet's geometry filter matches on INTERSECTS, not CONTAINS, and a
#    partial scene still costs the full 100 km2. Enforced as a post-filter in stage 3.
#
#    This is a CONTAINMENT test, not an area-ratio threshold. A ratio computed in
#    degree space carries up to 6e-4 of error at this latitude - the same order as the
#    1e-3 margin a ">= 99.9%" rule would try to discriminate on - and it admits scenes
#    that leave a corner of the AOI as nodata. Stage 3 projects to an equal-area CRS
#    and asks shapely whether the footprint covers the AOI outright.
#
#    Raise this only if you decide a small notch is acceptable; 0.0 means "no gaps".
COVERAGE_TOLERANCE_M2 = 0.0
# 2. Cloud, measured on the AOI via UDM2.
CLEAR_MIN    = 0.95
# 3. Vessel presence - set by eye in stage 7, over the WHOLE AOI (see stage 7).

# Optional extras. Neither gates anything.
GLINT_FILTER  = None    # e.g. 20 to drop hard-glinted scenes; None = keep all,
                        # since tile budget is ample enough to just look at them
ACOUSTIC_JOIN = False   # True to cross-reference the hydrophone record

# Duplicate handling is an EXPLICIT id list now, not groupby("date").head(1). Day
# granularity is too coarse: it would also drop one of 20210812_191832_32_2406 /
# 20210812_194340_68_1058, which are 25 MINUTES apart on different satellites and are
# two genuinely independent observations - a boat moves a long way in 25 min. The pair
# that is actually redundant is 30 SECONDS apart. See DROP_IDS below.
DEDUPE_BY_DAY = False

# ------------------------------------------------------- Option C selection
# Removed from the 30 gate-2 survivors before ordering, each with its reason.
DROP_IDS = {
    # Same overpass, 30 s apart, both PSB.SD - one observation, not two. Keeping the
    # marginally clearer and less-glinted of the pair (0.99799/31.06 vs 0.99780/31.95).
    "20210616_183140_94_2436",
    # The two thinnest survivors by aoi_clear. Cloud shadow and cloud edges are exactly
    # what fake a vessel for the NIR bright-on-dark-water rule in §7.
    "20230618_182410_36_24ab",   # 0.9737
    "20240701_193748_00_2482",   # 0.9747
}

# §5 acceptance test. Both assets of ONE acquisition are a controlled comparison of
# `visual` against RGB derived from analytic - same water, same boats, same instant.
# One per INSTRUMENT, because that is the axis that could shift the transform
# systematically; glint and water brightness already vary plenty within a scene.
PAIRED_VISUAL_IDS = [
    "20200730_192942_71_1059",   # PS2.SD, aoi_clear 1.000, glint 31.2. Chosen because
                                 # the handoff flags it as the scene with the clearest
                                 # apparent wakes - §5.4 compares DETECTION COUNTS, and
                                 # a control with no boats in it compares 0 against 0.
    "20250630_193824_56_24db",   # PSB.SD, aoi_clear 1.000, glint 28.5 - the most
                                 # glinted of the perfectly clear PSB.SD scenes, so it
                                 # stresses a fixed linear scale on bright water.
]

# Scenes in the analytic batch. Deliberately NOT SCENES_PER_MONTH, which is derived
# assuming one raster per scene while analytic_sr_udm2 may bill two. 27 analytic + 2
# visual = 29 slots = 2900 km2 IF a scene bills 100, leaving 100 km2 for a re-order -
# and there is no next month: the quota resets on the calendar and the project ends
# first. If the probe shows 200 km2 per analytic scene, this MUST come down.
BATCH_SIZE = 27

# ------------------------------------------------------------------ imagery quota
# AOI_AREA_KM2, CHARGE_PER_SCENE and SCENES_PER_MONTH are derived from the AOI
# geometry itself in the next cell, so they cannot drift from the geojson.
MIN_CHARGE    = 100.0
MONTHLY_QUOTA = 3000.0

# ------------------------------------------------------------------ tile quota
TILE_BUDGET    = 98249
TILE_ZOOM      = 15      # 3.15 m/px here, matching PlanetScope native 3 m
TILE_LEDGER    = WORK / "tile_ledger.json"

# The review queue, persisted. `df` lives only in kernel memory, so without this the
# reviewer can only run after rebuilding the whole gate chain. Stages 5 and 6 write it;
# the reviewer reads it, making a resumed review stage 1 + the reviewer cell.
SURVIVORS_CSV  = WORK / "gate2_survivors.csv"

# The review box is the AOI itself - see stage 7. There is no PREVIEW_HALF_M: gate 3
# must inspect exactly what gate 1 selected on.

def deg2tile(lon, lat, z):
    """WGS84 lon/lat -> slippy-map tile x/y. Defined here, not in the stage-2 probe,
    so stage 7 still works when the probe cell is skipped on a re-run."""
    n = 2 ** z
    return (int((lon + 180.0) / 360.0 * n),
            int((1.0 - math.asinh(math.tan(math.radians(lat))) / math.pi) / 2.0 * n))

# ------------------------------------------------------------------ bundle policy
# MEASURED 2026-08-27 against https://api.planet.com/compute/ops/bundles/spec, the
# same source the SDK validates against. For item type PSScene there are exactly TWO
# bundles carrying anything this project needs:
#
#   visual            -> ['ortho_visual']                                   1 raster
#   analytic_sr_udm2  -> ['ortho_analytic_4b_sr', '..._xml', 'ortho_udm2']  2 rasters
#
# The SINGLE_RASTER_BUNDLES set that used to sit here also listed "analytic_sr",
# "analytic_8b_sr", "analytic" and "analytic_8b". NONE of those exist for PSScene.
# The assert passed anyway because it only tested membership of a hand-written set,
# never the live spec, so it would have failed at order time instead of here.
#
# Option C (docs/plans/option_c_detection_plan.md §4) needs NIR on every scene it will
# certify as clean, and ortho_analytic_4b_sr is the only PSScene route to it. So the
# multi-raster bundle the old policy existed to avoid is now unavoidable, and the
# doubled-charge risk it was guarding against CANNOT be settled by assertion.
# It is settled empirically by the one-scene probe in stage 9: order one, read the
# quota_used delta in stage 10, and only then release the rest.
#
# Asset availability confirmed per item on BOTH instruments in the survivor pool via
# a live /assets call: ortho_analytic_4b_sr, ortho_udm2 and ortho_visual are present
# on PSB.SD and PS2.SD alike. Only the 8-band assets are PSB.SD-only; we do not use
# them (§4: "NIR is all we actually need", and 8-band may be licensed differently).
PSSCENE_BUNDLES = {"visual": 1, "analytic_sr_udm2": 2}    # name -> raster count
BUNDLE        = "analytic_sr_udm2"   # the NIR carrier, on every scene
PAIRED_BUNDLE = "visual"             # the §5 calibration control, 2 scenes only
for _b in (BUNDLE, PAIRED_BUNDLE):
    assert _b in PSSCENE_BUNDLES, (
        f"{_b!r} is not a PSScene bundle in the live spec. Re-check "
        f"https://api.planet.com/compute/ops/bundles/spec before ordering.")
# Never set fallback_bundle: it can silently substitute a different bundle.

print(f"Gates   : AOI fully contained, AOI clear >= {CLEAR_MIN:.0%}")
print(f"Bundles : {BUNDLE} on the batch, {PAIRED_BUNDLE} on {len(PAIRED_VISUAL_IDS)} paired")
print(f"Tiles   : {TILE_BUDGET:,} at z{TILE_ZOOM}")

### Authentication

Run `planet auth init` once in a CryoCloud terminal. That writes `~/.planet.json` with
restrictive permissions, so your key never enters a notebook cell or a git commit.

In [ ]:
try:
    auth = Auth.from_file()
    print("Authenticated from ~/.planet.json")
except Exception:
    if not os.environ.get("PL_API_KEY"):
        raise SystemExit("Run `planet auth init` in a terminal, or export PL_API_KEY.")
    auth = Auth.from_key(os.environ["PL_API_KEY"])
    print("Authenticated from PL_API_KEY")

API_KEY = auth.value        # for raw tile / thumbnail HTTP requests

In [ ]:
_nominal = json.load(open(AOI_PATH))
if   _nominal["type"] == "FeatureCollection": _nominal = _nominal["features"][0]["geometry"]
elif _nominal["type"] == "Feature":           _nominal = _nominal["geometry"]
assert _nominal["type"] == "Polygon"

_ring = _nominal["coordinates"][0]
_b = (min(c[0] for c in _ring), min(c[1] for c in _ring),
      max(c[0] for c in _ring), max(c[1] for c in _ring))
HYD_LON, HYD_LAT = -125.278277, 48.814200

# ------------------------------------------------------------------ quota inset
# The geojson is a nominal 10 x 10 km box measuring 100.00024 km2 - about 240 m2 OVER
# the 100 km2 the monthly quota divides into 30 slices of. Planet's clip-area
# accounting is exact, so 30 orders of the nominal box would ask for 3000.007 km2
# against a 3000 km2 quota and the 30th would be refused.
#
# Inset every edge to land under the line. The strict minimum is 6 mm (240 m2 over
# 40 km of perimeter), but that is not the number to use, for two reasons:
#
#   1. At 6 mm the equal-area and geodesic area models straddle 100 km2 (99.999999 vs
#      100.000030), so the bare minimum is not robust to which one Planet uses.
#   2. More importantly, "clip area" may not mean the AOI polygon at all. The AOI is a
#      lat/lon rectangle, so in any projected CRS its meridians converge and it becomes
#      a quadrilateral narrower at the north edge - its POLYGON area and its BOUNDING
#      BOX differ by ~90,000 m2. A clipped GeoTIFF is a rectangle. If Planet bills the
#      output footprint rather than the cut polygon, the bbox is the number that counts.
#
# At AOI_INSET_M = 5.0 the three candidate bases come out:
#   polygon                       99.800 km2   under
#   bounding box                  99.889 km2   under
#   bbox + full 3 m pixel snap   100.009 km2   marginally OVER
#
# So this covers the two likely bases with ~200,000 m2 of margin, but NOT the most
# pessimistic one, which assumes the clip window snaps outward a full pixel on all four
# edges (the expected case is half that). ~6 m would close even that gap. Settle it
# empirically instead of guessing: order one scene and read the quota_used delta from
# stage 10 - one scene costs 1/30th of a month and turns this into a fact.
AOI_INSET_M = 5.0

_dlat = AOI_INSET_M / 111_206.0
_dlon = AOI_INSET_M / (111_320.0 * math.cos(math.radians(HYD_LAT)))
AOI_BBOX = (_b[0] + _dlon, _b[1] + _dlat, _b[2] - _dlon, _b[3] - _dlat)
aoi = {"type": "Polygon", "coordinates": [[
    [AOI_BBOX[0], AOI_BBOX[1]], [AOI_BBOX[2], AOI_BBOX[1]],
    [AOI_BBOX[2], AOI_BBOX[3]], [AOI_BBOX[0], AOI_BBOX[3]],
    [AOI_BBOX[0], AOI_BBOX[1]]]]}

# `aoi` is now THE aoi: the search filter, the UDM2 screen, gate 1 containment, the
# preview box and - critically - clip_tool() in stage 9 all use it, so what gets
# billed is exactly what gets gated on.

# ------------------------------------------------------------------ metric AOI
# Everything that compares areas or distances works on this projection, not on
# degrees. Lambert azimuthal equal-area centred on the hydrophone: areas are exact
# and the origin is the instrument, so the centring assert below is meaningful.
AOI_LAEA = f"+proj=laea +lat_0={HYD_LAT} +lon_0={HYD_LON} +datum=WGS84 +units=m"
_to_m    = Transformer.from_crs("EPSG:4326", AOI_LAEA, always_xy=True).transform
AOI_M    = _transform(_to_m, shape(aoi))

AOI_AREA_KM2     = AOI_M.area / 1e6
CHARGE_PER_SCENE = max(AOI_AREA_KM2, MIN_CHARGE)   # the 100 km2 clip minimum applies
SCENES_PER_MONTH = int(MONTHLY_QUOTA // CHARGE_PER_SCENE)

# The projected bounding box - the number that matters if Planet bills the clipped
# raster footprint rather than the cut polygon. Tracked so the assumption is visible.
_pb = AOI_M.bounds
AOI_BBOX_KM2 = (_pb[2] - _pb[0]) * (_pb[3] - _pb[1]) / 1e6

# Assert the properties the study depends on, rather than trusting constants to still
# match the geojson on disk.
_c   = AOI_M.centroid
_off = math.hypot(_c.x, _c.y)
assert _off < 50.0,           f"AOI centre is {_off:.0f} m from the hydrophone"
assert AOI_AREA_KM2 < 100.0,  f"AOI polygon is {AOI_AREA_KM2:.6f} km2, not under the minimum"
assert AOI_BBOX_KM2 < 100.0,  f"AOI bbox is {AOI_BBOX_KM2:.6f} km2, not under the minimum"
assert SCENES_PER_MONTH * CHARGE_PER_SCENE <= MONTHLY_QUOTA, "monthly batch exceeds quota"

print(f"AOI bbox   : {tuple(round(v, 6) for v in AOI_BBOX)}")
print(f"AOI extent : {(_pb[2]-_pb[0])/1000:.4f} x {(_pb[3]-_pb[1])/1000:.4f} km, "
      f"centred {_off:.1f} m from the hydrophone")
print(f"AOI area   : polygon {AOI_AREA_KM2:.5f} km^2 | bounding box {AOI_BBOX_KM2:.5f} km^2 "
      f"(inset {AOI_INSET_M} m/edge)")
print(f"             {(100.0-AOI_AREA_KM2)*1e6:,.0f} m^2 under the 100 km^2 clip minimum "
      f"on polygon, {(100.0-AOI_BBOX_KM2)*1e6:,.0f} m^2 on bbox")
print(f"Imagery    : {SCENES_PER_MONTH} scenes/month at {CHARGE_PER_SCENE:.2f} km^2 "
      f"= {SCENES_PER_MONTH*CHARGE_PER_SCENE:,.1f} of {MONTHLY_QUOTA:,.0f} km^2")

## 2. Search — deliberately loose

Six separate June–August windows joined with an OR filter. A single
`gte=2020-06-01, lte=2025-08-31` range would sweep in every winter in between.

`cloud_cover` is a **fraction**, not a percent — passing `10` instead of `0.10` matches
everything ever acquired.

In [ ]:
search_filter = data_filter.and_filter([
    data_filter.geometry_filter(aoi),
    data_filter.range_filter("cloud_cover", lte=SEARCH_CLOUD_MAX),
    data_filter.or_filter([
        data_filter.date_range_filter("acquired",
                                      gte=datetime(y, SUMMER[0], 1),
                                      lt =datetime(y, SUMMER[1], 1))
        for y in YEARS]),
    data_filter.permission_filter(),
    data_filter.string_in_filter("quality_category", ["standard"]),
])

async def run_search():
    async with Session(auth=auth) as sess:
        res = sess.client("data").search([ITEM_TYPE],
                                         search_filter=search_filter, limit=0)
        return [i async for i in res]

items = await run_search()
json.dump(items, open(WORK / "search_results.json", "w"))
print(f"{len(items)} candidates cached (search costs no quota)")

In [ ]:
df = pd.DataFrame([{
    "id":            it["id"],
    "acquired":      pd.to_datetime(it["properties"]["acquired"]),
    "cloud_cover":   it["properties"]["cloud_cover"],
    "clear_percent": it["properties"].get("clear_percent"),
    "instrument":    it["properties"].get("instrument"),
    "sun_elevation": it["properties"].get("sun_elevation"),
    "sun_azimuth":   it["properties"].get("sun_azimuth"),
    "view_angle":    it["properties"].get("view_angle"),
    "sat_azimuth":   it["properties"].get("satellite_azimuth"),
    "thumbnail":     it["_links"].get("thumbnail"),
    "tiles_link":    it["_links"].get("tiles"),
} for it in items])

df["date"] = df["acquired"].dt.date
df["year"] = df["acquired"].dt.year
df = df.sort_values("acquired").reset_index(drop=True)

print(df.groupby("year").agg(scenes=("id","size"), days=("date","nunique")))
print(f"\ntiles link present on {df['tiles_link'].notna().sum()}/{len(df)} items")
if df["tiles_link"].isna().all():
    print("  -> set TILE_URL_TEMPLATE in stage 7; inspect items[0]['_links'] first")

print(json.dumps(items[0]["_links"], indent=2))

In [ ]:
# Route probe. Items carry no "_links.tiles" on this plan, so the tile route has to be
# supplied by hand. Probing with an arbitrary df.iloc[0] is NOT a valid test: that scene
# may not cover the target, and the tile service answers 200 with a transparent PNG for
# any tile outside a scene footprint. Probe with a scene that CONTAINS the AOI, and read
# the ALPHA CHANNEL, not just the status code.
#
# Diagnostic only - safe to skip on a re-run. deg2tile lives in stage 1.
from shapely.geometry import shape as _shape

# A scene that fully contains the AOI, so a blank result means a broken route.
_covering = [it for it in items
             if _transform(_to_m, _shape(it["geometry"])).buffer(0).covers(AOI_M)]
assert _covering, "no scene contains the AOI - check the AOI or widen the search"
_probe = _covering[0]
_pid   = _probe["id"]
z      = TILE_ZOOM
x, y   = deg2tile(HYD_LON, HYD_LAT, z)

# Far outside the footprint: the negative control that shows what "no data" looks like.
_b = _shape(_probe["geometry"]).bounds
xo, yo = deg2tile(_b[2] + 0.20, _b[3] + 0.20, z)

candidates = [
    ("tiles.planet  /data/v1/PSScene/{id}/{z}/{x}/{y}.png",
     f"https://tiles.planet.com/data/v1/PSScene/{_pid}/{z}/{{X}}/{{Y}}.png"),
    ("tiles.planet  .../item-types/PSScene/items/{id}/tile/...",
     f"https://tiles.planet.com/data/v1/item-types/PSScene/items/{_pid}/tile/{z}/{{X}}/{{Y}}.png"),
    ("tiles.planet  .../item-types/PSScene/items/{id}/tiles/...",
     f"https://tiles.planet.com/data/v1/item-types/PSScene/items/{_pid}/tiles/{z}/{{X}}/{{Y}}.png"),
    ("api.planet    /data/v1/PSScene/{id}/{z}/{x}/{y}.png",
     f"https://api.planet.com/data/v1/PSScene/{_pid}/{z}/{{X}}/{{Y}}.png"),
]

def _describe(r):
    """Status code alone cannot tell imagery from nodata - decode and look."""
    if r.status_code != 200 or "image" not in r.headers.get("content-type", ""):
        return f"{r.status_code} {len(r.content):>7}B  {r.text[:40]!r}"
    a = np.array(Image.open(BytesIO(r.content)).convert("RGBA"))
    cov = (a[..., 3] > 0).mean()
    p99 = int(np.percentile(a[..., :3][a[..., 3] > 0], 99)) if cov else -1
    kind = "NODATA" if cov < 0.01 else ("flat" if p99 < 25 else "IMAGERY")
    return f"200 {len(r.content):>7}B  alpha={cov:6.1%}  p99={p99:>3}  {kind}"

print(f"probe scene {_pid}  (contains the AOI)\n")
with httpx.Client(auth=(API_KEY, ""), timeout=30, follow_redirects=True) as c:
    for label, tpl in candidates:
        for tag, (tx, ty) in (("on-target ", (x, y)), ("off-scene ", (xo, yo))):
            u = tpl.replace("{X}", str(tx)).replace("{Y}", str(ty))
            try:
                print(f"  {label[:44]:44s} {tag} {_describe(c.get(u))}")
            except Exception as e:
                print(f"  {label[:44]:44s} {tag} ERR {type(e).__name__}")
    print()

# Result on this account: only the first route serves imagery. The off-scene control
# returns 200 with an 820-byte fully transparent PNG - which is why a naive
# "did I get a 200?" check passes while the mosaic comes back black.

## 3. Gate 1 — coverage

`geometry_filter` matches on **intersects**, so a scene clipping one corner of the AOI comes
back looking identical to one covering it whole. Planet offers no "contains" filter, so this
is enforced here.

A scene covering 30% of the AOI is charged the same 100 km² as a full one — the worst value
available — and leaves nodata over exactly the water you care about.

**Containment, not a coverage percentage.** The gate is `footprint.covers(AOI)` in an
equal-area projection, not `intersection.area / aoi.area >= 0.999`. Two reasons:

1. A 99.9% threshold still admits ~0.1 km² of nodata. On this archive exactly two scenes sit
   in that band — missing ~9 k and ~18 k m² — and both are reported by the cell below
   rather than silently dropped.
2. An area ratio taken in degree space carries up to 6×10⁻⁴ of error at 48.8°N, the same
   order as the 10⁻³ margin such a threshold discriminates on. The test would not be precise
   enough to enforce its own rule.

Set `COVERAGE_TOLERANCE_M2` above 0 in stage 1 if you decide a small notch is acceptable.

The footprint polygon is already in the cached results, so this costs nothing.

In [ ]:
def coverage_stats(geom):
    """(fraction, missing_m2, covers) for a footprint against the AOI.

    Computed in the equal-area projection from stage 1. `covers` is the actual gate:
    a topological containment test, exact, with no threshold to tune. The fraction and
    the missing area are reported so a near-miss is legible rather than just absent."""
    s     = _transform(_to_m, shape(geom)).buffer(0)
    inter = s.intersection(AOI_M).area
    return inter / AOI_M.area, AOI_M.area - inter, s.covers(AOI_M)

_stats = {it["id"]: coverage_stats(it["geometry"]) for it in items}
df["aoi_coverage"]   = df["id"].map(lambda i: _stats[i][0])
df["aoi_missing_m2"] = df["id"].map(lambda i: _stats[i][1])
df["aoi_covered"]    = df["id"].map(lambda i: _stats[i][2])

full   = df["aoi_covered"] | (df["aoi_missing_m2"] <= COVERAGE_TOLERANCE_M2)
sliver = df["aoi_coverage"] < 0.5
print(f"full containment : {full.sum():4d}")
print(f"partial          : {((~full) & ~sliver).sum():4d}")
print(f"slivers (<50%)   : {sliver.sum():4d}")
print(f"\nquota slivers would waste: {sliver.sum()*CHARGE_PER_SCENE:,.0f} km^2 charged for "
      f"{df.loc[sliver,'aoi_coverage'].sum()*AOI_AREA_KM2:.0f} km^2 of usable pixels")

# The scenes an area-ratio threshold would have let through. Each leaves a corner of
# the AOI as nodata - over exactly the water the 100 km2 minimum charge is buying.
near = df[~full & (df["aoi_coverage"] >= 0.99)]
if len(near):
    print(f"\nrejected despite >=99% overlap (this is the point of the gate):")
    for _, r in near.sort_values("aoi_coverage", ascending=False).iterrows():
        print(f"  {r['id']}  {r['aoi_coverage']:.6f}  missing {r['aoi_missing_m2']:>10,.0f} m^2")

before = len(df)
df = df[full].copy()
print(f"\nGate 1: {before} -> {len(df)} scenes fully containing the AOI")

## 4. Gate 2 — cloud, measured on the AOI

Planet's documentation states that downloading UDM2 via the **Data API** does not count
against download quota. That is what makes this affordable.

`cloud_cover` and `clear_percent` are computed over the whole ~637 km² footprint, and your
AOI is ~15% of it, so the scene-level number tells you little. A 55%-clear scene may be
spotless over Folger Passage; a 96%-clear scene can have its one cloud sitting on the
hydrophone.

Single-band windowed reads keep peak memory near 10 MB regardless of scene size.

In [ ]:
UDM2_CLEAR_BAND = 1   # 1 clear, 2 snow, 3 shadow, 4 light haze,
                      # 5 heavy haze, 6 cloud, 7 confidence, 8 unusable-data mask

def aoi_clear_fraction(udm_path, aoi_geojson):
    '''Fraction of AOI pixels flagged clear.
    boundless=True pads outside the raster with 0 (= not clear), so a scene with
    partial coverage scores low honestly rather than returning a truncated array
    that misaligns against the geometry mask.'''
    with rasterio.open(udm_path) as src:
        geom = transform_geom("EPSG:4326", src.crs, aoi_geojson)
        win  = from_bounds(*geom_bounds(geom), src.transform)
        win  = win.round_offsets().round_lengths()
        clear = src.read(UDM2_CLEAR_BAND, window=win, boundless=True, fill_value=0)
        if clear.size == 0:
            return np.nan
        inside = geometry_mask([geom], out_shape=clear.shape,
                               transform=src.window_transform(win), invert=True)
        return float((clear[inside] == 1).mean()) if inside.any() else np.nan


# Gate 2 is the slowest free step: one asset activation + download per scene. Cache the
# scores so a re-run, a kernel restart, or an interrupted loop does not re-download
# masks that were already measured. Delete this file to force a fresh screen.
AOI_CLEAR_CACHE = WORK / "aoi_clear.json"

def _load_clear():
    return json.load(open(AOI_CLEAR_CACHE)) if AOI_CLEAR_CACHE.exists() else {}

def _save_clear(d):
    """Write via temp + rename. An in-place truncate leaves a corrupt cache if the
    process is killed mid-write, which for a multi-hour loop is a real risk."""
    tmp = AOI_CLEAR_CACHE.with_suffix(".tmp")
    json.dump({k: (None if v != v else v) for k, v in d.items()},
              open(tmp, "w"), indent=1)
    tmp.replace(AOI_CLEAR_CACHE)

UDM2_CONCURRENCY = 8

async def screen_udm2(item_ids, directory=None, keep_files=False, use_cache=True,
                      concurrency=UDM2_CONCURRENCY):
    '''Download + score UDM2. Costs no imagery quota.

    Activation is the bottleneck, not bandwidth: Planet generates each ortho_udm2 on
    demand and a cold asset takes minutes to become available. Screening scenes one at
    a time pays that wait 48 times end to end - measured at ~4.9 min/scene, ~4 h for
    the pool. The wait is idle polling, not work, so overlapping scenes collapses it:
    at concurrency 8 the pool finished in 27 min.

    Files are deleted after scoring unless keep_files, to protect CryoCloud disk.
    Results are cached and flushed atomically after every scene, so the run is
    resumable and a kill mid-flight costs at most the scenes still in flight.'''
    directory = directory or UDM_DIR
    out  = _load_clear() if use_cache else {}
    todo = [i for i in item_ids if i not in out]
    if use_cache and len(todo) < len(item_ids):
        print(f"  {len(item_ids)-len(todo)} cached, {len(todo)} to screen")
    if not todo:
        return {i: out[i] for i in item_ids}

    sem, done = asyncio.Semaphore(concurrency), 0

    async def one(cl, iid):
        nonlocal done
        async with sem:
            try:
                a = await cl.get_asset(ITEM_TYPE, iid, "ortho_udm2")
                await cl.activate_asset(a)
                a = await cl.wait_asset(a, max_attempts=200)
                p = await cl.download_asset(a, directory=directory,
                                            overwrite=False, progress_bar=False)
                # Blocking rasterio read inside the loop. It is a windowed single-band
                # read (~10 MB, sub-second) against multi-minute activation waits, so
                # it is not worth moving to a thread.
                out[iid] = aoi_clear_fraction(p, aoi)
                if not keep_files:
                    Path(p).unlink(missing_ok=True)
            except Exception as e:
                print(f"  {iid}: {type(e).__name__} {e}", flush=True)
                out[iid] = np.nan
            done += 1
            if use_cache:
                _save_clear(out)
            if done % 5 == 0 or done == len(todo):
                print(f"  screened {done}/{len(todo)}", flush=True)

    async with Session(auth=auth) as sess:
        cl = sess.client("data")
        await asyncio.gather(*[one(cl, i) for i in todo])
    return {i: out.get(i, np.nan) for i in item_ids}

_clear = await screen_udm2(df["id"].tolist())
df["aoi_clear"] = df["id"].map(lambda i: np.nan if _clear.get(i) is None else _clear[i])

In [ ]:
passes = df["aoi_clear"] >= CLEAR_MIN
print(f"AOI-clear vs scene clear_percent correlation: "
      f"{df[['aoi_clear','clear_percent']].corr().iloc[0,1]:.2f}")
print(f"recovered (clear on AOI, but cloud_cover > 0.10): "
      f"{(passes & (df['cloud_cover'] > 0.10)).sum()}")
print(f"rejected  (cloud_cover <= 0.10, but not clear on AOI): "
      f"{((df['cloud_cover'] <= 0.10) & ~passes).sum()}")

before = len(df)
df = df[passes].copy()
print(f"\nGate 2: {before} -> {len(df)} scenes >= {CLEAR_MIN:.0%} clear over the AOI")

## 5. Glint — review-queue ordering only

Glint is deterministic geometry, computable from metadata for free. It is **not** a gate here:
your tile budget comfortably exceeds the candidate pool, so there is no reason to discard
scenes you could simply look at. It is used to order the review queue so the scenes most
likely to show something come first.

$$\cos\Theta_g = \cos\theta_v\cos\theta_s - \sin\theta_v\sin\theta_s\cos(\phi_v-\phi_s)$$

Small $\Theta_g$ means the sensor is staring into the sun's glitter pattern, which flattens
wake contrast. Set `GLINT_FILTER` to a number only if you find glinted scenes are wasting
your review time.

In [ ]:
ts   = np.radians(90.0 - df["sun_elevation"])
tv   = np.radians(df["view_angle"].abs())
dphi = np.radians(df["sat_azimuth"] - df["sun_azimuth"])
df["glint_angle"] = np.degrees(np.arccos(np.clip(
    np.cos(tv)*np.cos(ts) - np.sin(tv)*np.sin(ts)*np.cos(dphi), -1, 1)))

print(df["glint_angle"].describe().round(1))

if GLINT_FILTER is not None:
    before = len(df)
    df = df[df["glint_angle"] > GLINT_FILTER].copy()
    print(f"\nglint filter: {before} -> {len(df)}")

# Best-looking first: clearest, then least glinted.
df = df.sort_values(["aoi_clear", "glint_angle"], ascending=False).reset_index(drop=True)
df.to_csv(SURVIVORS_CSV, index=False)
print(f"\n{len(df)} scenes queued for review -> {SURVIVORS_CSV}")

## 6. Quicklook skim — free

Thumbnails run ~100 m/pixel, so no boat is visible. What they do show is marine fog banks and
broad glint sheets, both of which UDM2 misreads as clear over water. A quick pass here saves
tile quota and review time.

In [ ]:
def contact_sheet(rows, url_col="thumbnail", cols=6, thumb=200):
    imgs = []
    with httpx.Client(auth=(API_KEY, ""), timeout=30, follow_redirects=True) as c:
        for _, r in rows.iterrows():
            if not r.get(url_col):
                continue
            try:
                imgs.append(Image.open(BytesIO(c.get(r[url_col]).content))
                            .convert("RGB").resize((thumb, thumb)))
            except Exception:
                pass
    if not imgs:
        return None
    nrow = -(-len(imgs) // cols)
    sheet = Image.new("RGB", (cols*thumb, nrow*thumb), "black")
    for i, im in enumerate(imgs):
        sheet.paste(im, ((i % cols)*thumb, (i // cols)*thumb))
    return sheet

contact_sheet(df.head(36))

In [ ]:
FOGGED = []          # scene ids that are obviously fogged or glinted
df = df[~df["id"].isin(FOGGED)].copy()
df.to_csv(SURVIVORS_CSV, index=False)      # keep the persisted queue in step with df
print(f"{len(df)} scenes after visual skim")

## 7. Gate 3 — tile preview and vessel review

Tiles are metered separately from the imagery quota, so this is free against the 30-scene
budget. At z15 the ground resolution is 3.15 m/px, matching PlanetScope native.

| Object | Pixels at z15 |
|---|---|
| 7 m skiff | 2.2 |
| 12 m charter | 3.8 |
| 30 m coastal freighter | 9.5 |
| Planing wake, width | 19 |
| Planing wake, length | 127 |

This resolves **wakes and mid-size hulls**, not small boats sitting still.

**The review box is the whole AOI**, not an inner crop. Gate 1 selects scenes on full
containment of the AOI, so reviewing a 5 × 5 km centre box would mean choosing scenes on
100 km² and inspecting 25 km² of them — a vessel in the outer 75% would score as
"no vessel". At z15 the AOI is a 14 × 14 grid, 196 tiles per scene, ~500 previews affordable
per month against a candidate pool in the dozens. The tile budget is not the constraint;
wall-clock is.

Tiles are 8-bit RGB rendered with a stretch tuned for land, and over dark water that crushes
wake contrast toward black — measured here, the water occupies roughly DN 0–38 of 255. Stage
7 therefore stretches each mosaic over its valid pixels before saving, and keeps nodata
black so it cannot be mistaken for dark water.

Previews are saved at **native resolution** (3584 px, 3.15 m/px). Downsampling them to fit a
screen is what destroys the detection margin the table above promises — see
`preview_batch`'s docstring.

In [ ]:
# Verified against this account in stage 2's probe: this is the Data API XYZ tile route.
# It needs NO asset activation - every asset on these items reports status="inactive"
# and tiles still render. Confirmed 200-with-pixels on all coverage-gated scenes,
# 2020-2025, both PS2.SD and PSB.SD. tiles0-3.planet.com are equivalent shards.
TILE_URL_TEMPLATE = "https://tiles.planet.com/data/v1/PSScene/{item_id}/{z}/{x}/{y}.png"

# Three distinct tile states, separable only after decoding. Byte size alone is a good
# first-pass tell because a constant-valued PNG compresses to almost nothing:
#   ~820 B, alpha == 0        -> NODATA, tile outside the scene footprint
#   1-5 kB, alpha == 1, p99<25 -> opaque but flat; low contrast
#   30-140 kB                  -> real imagery
#
# NOTE: "flat" is measured per tile on the RAW DN, while the stretch is applied globally
# across the mosaic afterwards. A flat tile can still show a bright wake once stretched,
# so flat means "low overall contrast", NOT "nothing detectable". Do not gate on it.
NODATA_ALPHA_FRAC = 0.01   # below this, treat the tile as outside the footprint
FLAT_P99          = 25     # 99th pct DN over valid pixels

def tile_grid(bbox, z):
    x0, y0 = deg2tile(bbox[0], bbox[3], z)
    x1, y1 = deg2tile(bbox[2], bbox[1], z)
    return list(range(x0, x1 + 1)), list(range(y0, y1 + 1))

# THE REVIEW BOX IS THE AOI. Gate 1 selects scenes on full containment of the AOI, so
# gate 3 must look at the whole AOI too - otherwise scenes are chosen on 100 km2 and
# reviewed on a fraction of it, and a vessel outside the reviewed part reads as "no
# vessel". Taken from AOI bounds directly: no cos(lat) round-trip, no half-width to
# keep in sync.
PREVIEW_BOX = shape(aoi).bounds
xs, ys = tile_grid(PREVIEW_BOX, TILE_ZOOM)
per_scene = len(xs) * len(ys)

from shapely.geometry import box as _box
_seen = _box(*PREVIEW_BOX).intersection(shape(aoi)).area / shape(aoi).area
print(f"review box -> {len(xs)}x{len(ys)} = {per_scene} tiles/scene, "
      f"covering {_seen:.1%} of the AOI")
print(f"z{TILE_ZOOM} ground resolution here: "
      f"{(360/2**TILE_ZOOM)*111320*math.cos(math.radians(HYD_LAT))/256:.2f} m/px")
print(f"affordable previews: {TILE_BUDGET // per_scene:,}   candidates: {len(df)}")
print(f"this pool costs {len(df)*per_scene:,} tiles "
      f"({len(df)*per_scene/TILE_BUDGET:.1%} of the monthly tile budget)")

In [ ]:
def _ledger():
    if TILE_LEDGER.exists():
        return json.load(open(TILE_LEDGER))
    return {"month": datetime.now().strftime("%Y-%m"), "used": 0}

def _spend(n):
    led, now = _ledger(), datetime.now().strftime("%Y-%m")
    if led["month"] != now:            # tile quota resets monthly, no rollover
        led = {"month": now, "used": 0}
    led["used"] += n
    json.dump(led, open(TILE_LEDGER, "w"))
    return led

def tile_url(row, z, x, y):
    tpl = row.get("tiles_link") or TILE_URL_TEMPLATE
    if not tpl:
        raise RuntimeError("No tile URL: set TILE_URL_TEMPLATE in this stage.")
    return (tpl.replace("{item_id}", row["id"]).replace("{z}", str(z))
               .replace("{x}", str(x)).replace("{y}", str(y)).replace("{0}", "0"))


def classify_tile(arr):
    """NODATA / flat / imagery. A 200 proves only that the request was well-formed:
    the tile service returns a transparent PNG for anything off-footprint."""
    valid = arr[..., 3] > 0
    cov = float(valid.mean())
    if cov < NODATA_ALPHA_FRAC:
        return "nodata", cov, -1
    p99 = int(np.percentile(arr[..., :3][valid], 99))
    return ("flat" if p99 < FLAT_P99 else "imagery"), cov, p99


async def fetch_mosaic(row, bbox, z, concurrency=3, tile_px=256):
    """Fetch tiles for bbox and composite. Returns (RGBA sheet, qc dict).

    concurrency=3 deliberately. At 6 the tile service starts returning 503 under
    sustained load and the rate worsens as a batch progresses - 16 of 48 scenes lost
    2-20 tiles each. At 3 the same scenes came back clean, for only a modest wall-clock
    cost. The fix is to stop provoking it, not just to retry harder."""
    xs, ys = tile_grid(bbox, z)
    sheet  = Image.new("RGBA", (len(xs)*tile_px, len(ys)*tile_px))
    sem    = asyncio.Semaphore(concurrency)
    qc     = {"requested": len(xs)*len(ys), "served": 0,
              "imagery": 0, "flat": 0, "nodata": 0, "failed": 0}

    tile_url(row, z, xs[0], ys[0])     # fail fast on a bad template, before any spend

    async def one(client, i, x, j, y):
        async with sem:
            url = tile_url(row, z, x, y)
            for attempt in range(6):
                try:
                    r = await client.get(url)
                except Exception:
                    await asyncio.sleep(2 ** attempt); continue
                # 429 (rate limited) and 5xx (server overload) are BOTH transient. The
                # tile service returns 503 under sustained load, and treating that as
                # fatal drops tiles into the mosaic as black squares indistinguishable
                # from nodata - which is exactly the silent failure this stage exists
                # to prevent. Only a 4xx other than 429 is a real, permanent error.
                if r.status_code == 429 or r.status_code >= 500:
                    await asyncio.sleep(2 ** attempt); continue
                if r.status_code != 200:
                    qc["failed"] += 1
                    print(f"    {r.status_code} {url.rsplit('/data/v1/',1)[-1]}")
                    return
                qc["served"] += 1
                im  = Image.open(BytesIO(r.content)).convert("RGBA")
                arr = np.array(im)
                kind, _, _ = classify_tile(arr)
                qc[kind] += 1
                if kind != "nodata":
                    sheet.paste(im, (i*tile_px, j*tile_px))
                return
            qc["failed"] += 1          # retries exhausted - say so, do not fail silently
            print(f"    gave up after 6 attempts: {url.rsplit('/data/v1/',1)[-1]}")

    async with httpx.AsyncClient(auth=(API_KEY, ""), timeout=60,
                                 follow_redirects=True) as client:
        await asyncio.gather(*[one(client, i, x, j, y)
                               for i, x in enumerate(xs) for j, y in enumerate(ys)])
    _spend(qc["served"])               # count tiles the service actually served
    return sheet, qc


def stretch(sheet, lo_pct=1.0, hi_pct=99.5):
    """Percentile stretch over valid pixels only.

    The tile service renders an 8-bit product tuned for land. Over Folger Passage the
    water occupies roughly DN 0-38 of 255 - a raw preview is visually black and a
    reviewer will score every scene 'no vessel'. Nodata stays black so it cannot be
    mistaken for dark water."""
    a     = np.array(sheet)
    valid = a[..., 3] > 0
    if not valid.any():
        return Image.fromarray(a[..., :3]), (0, 0)
    v      = a[..., :3][valid]
    lo, hi = np.percentile(v, lo_pct), np.percentile(v, hi_pct)
    out    = np.clip((a[..., :3].astype(np.float32) - lo) / max(hi - lo, 1) * 255,
                     0, 255).astype(np.uint8)
    out[~valid] = 0
    return Image.fromarray(out), (float(lo), float(hi))


async def preview_batch(rows, bbox, z, max_px=None, quality=95):
    """One stretched JPEG per scene at NATIVE tile resolution, tiles discarded.

    max_px=None means no downsampling. This matters more than it looks: the full-AOI
    mosaic is 3584 px, so an earlier max_px=1400 threw away 2.56x, taking the preview
    from 3.15 m/px to 8.05 m/px. At that scale a 12 m charter falls from 3.8 px to 1.5
    and a 7 m skiff from 2.2 to 0.9 - i.e. the resolution the stage 7 table promises
    describes the tiles, not the file you would actually review. Set max_px only if you
    knowingly want a smaller file.

    Size at native resolution, measured on this AOI: 2-9 MB/scene at quality=95,
    varying with how much structure the scene carries (flat water compresses; cloud and
    surf do not). The 48-scene pool came to 226 MB. quality=90 roughly halves the file
    but raises RMS compression error from 2.7 to 4.2 DN, a poor trade when the target is
    a 2-4 px object. PNG is lossless at ~27 MB/scene.

    Scenes whose tiles carry no usable contrast are reported, not silently saved as a
    black square."""
    out, skipped = [], []
    for n, (_, r) in enumerate(rows.iterrows(), 1):
        try:
            sheet, qc = await fetch_mosaic(r, bbox, z)
            if qc["imagery"] == 0:
                skipped.append((r["id"], qc))
                print(f"  {r['id']}: no usable tiles {qc}")
                continue
            im, (lo, hi) = stretch(sheet)
            if max_px:
                im.thumbnail((max_px, max_px))
            p = PREVIEW / f"{r['id']}.jpg"
            im.save(p, "JPEG", quality=quality)
            out.append(p)
            if qc["imagery"] < qc["requested"]:
                print(f"  {r['id']}: {qc['imagery']}/{qc['requested']} imagery "
                      f"(nodata {qc['nodata']}, flat {qc['flat']}, failed {qc['failed']})")
        except Exception as e:
            print(f"  {r['id']}: {type(e).__name__} {e}")
        if n % 20 == 0:
            print(f"  {n}/{len(rows)}   tiles used: {_ledger()['used']:,}")
    if skipped:
        print(f"\n{len(skipped)} scene(s) produced no reviewable preview")
    return out

**Test on one scene before running the batch.** Tiles are 8-bit RGB rendered with a stretch
tuned for land, and over dark water that can crush wake contrast toward black. Pick a date
with a known close vessel passage and confirm you can see it; otherwise a page of
empty-looking previews will read as "no boats" when it means "wrong stretch".

In [ ]:
test = df.iloc[[0]]
paths = await preview_batch(test, PREVIEW_BOX, TILE_ZOOM)
Image.open(paths[0]) if paths else print("no preview produced - check TILE_URL_TEMPLATE")

In [ ]:
paths = await preview_batch(df, PREVIEW_BOX, TILE_ZOOM)
led = _ledger()
print(f"\n{len(paths)} previews in {PREVIEW}")
print(f"Tiles used: {led['used']:,} / {TILE_BUDGET:,} "
      f"({TILE_BUDGET - led['used']:,} left this month)")

### Record what you see

Run the reviewer below. It pages through the saved previews one scene at a time and writes
your calls straight to `planet_folger/review_labels.json`, so nothing is hand-transcribed —
scene ids like `20210601_191849_05_2412` never have to be typed, and a mistyped character
can't silently drop a scene from the order.

**The overview is context, not the review.** It is downsampled to fit the screen, where a
12 m hull is about one pixel. Detection happens in the panel view: a 1:1 native-resolution
crop, 2.8 × 2.8 km per panel at 3.15 m/px. The panel buttons track which parts of the AOI you
have actually opened, so a "no vessel" call made after looking at 2 of 16 panels is visible
as exactly that — the next cell lists them rather than letting them pass as negatives.

A **Vessel** call is self-validating: you saw one, and the other panels don't matter. A
**No vessel** call is a claim about the whole 100 km², so it is only as good as the area you
actually opened. Negatives are the expensive calls here, not positives.

Two suggestions, unchanged. Use **Unsure** rather than forcing a binary call on an ambiguous
three-pixel blob — you can decide later whether to spend slots on those. And if you plan to
run the acoustic join, label these **before** looking at the hydrophone record, or you will
unconsciously find boats in scenes you already know were noisy.

Your labels persist across kernel restarts, so the review can be done in several sittings.
The reviewer reads the queue from `gate2_survivors.csv` when `df` is not in memory, so
resuming later is stage 1 plus the reviewer cell — no auth, no search, no rebuild.

In [ ]:
import ipywidgets as W
from IPython.display import display

REVIEW_LABELS = WORK / "review_labels.json"
PANEL_GRID    = 4      # 4 x 4 panels per scene -> 896 px each, ~2.8 km at 3.15 m/px
OVERVIEW_PX   = 760    # context only; NOT the review resolution

def _load_review():
    if REVIEW_LABELS.exists():
        d = json.load(open(REVIEW_LABELS))
        return d.get("labels", {}), {k: set(v) for k, v in d.get("seen", {}).items()}
    return {}, {}

def _save_review(labels, seen):
    json.dump({"labels": labels, "seen": {k: sorted(v) for k, v in seen.items()}},
              open(REVIEW_LABELS, "w"), indent=1)

def _png_bytes(img):
    b = BytesIO(); img.save(b, "PNG"); return b.getvalue()


class Reviewer:
    """Page through full-resolution previews and label them without typing ids.

    Decisions are written to REVIEW_LABELS on every click, so a kernel restart loses
    nothing and the review can be done across several sittings."""

    LABELS = [("vessel", "Vessel",    "success"),
              ("unsure", "Unsure",    "warning"),
              ("none",   "No vessel", "")]

    def __init__(self, rows, preview_dir=PREVIEW, grid=PANEL_GRID, overview_px=OVERVIEW_PX):
        self.grid, self.overview_px = grid, overview_px
        self.rows = [r for _, r in rows.iterrows()
                     if (preview_dir / f"{r['id']}.jpg").exists()]
        self.missing = len(rows) - len(self.rows)
        self.dir = preview_dir
        self.labels, self.seen = _load_review()
        self.i, self._img = 0, None

        self.status  = W.HTML()
        self.overview= W.Image(format="png")
        self.panel   = W.Image(format="png")
        self.panel_caption = W.HTML()

        self.panel_btns = []
        rows_box = []
        for r in range(grid):
            row = []
            for c in range(grid):
                b = W.Button(description=f"{r*grid+c+1}",
                             layout=W.Layout(width="46px", height="30px"))
                b.on_click(self._make_panel_cb(r*grid + c))
                row.append(b); self.panel_btns.append(b)
            rows_box.append(W.HBox(row))
        self.panel_grid_box = W.VBox(rows_box)

        prev = W.Button(description="< Prev", layout=W.Layout(width="90px"))
        nxt  = W.Button(description="Next >", layout=W.Layout(width="90px"))
        prev.on_click(lambda _: self._go(self.i - 1))
        nxt.on_click(lambda _: self._go(self.i + 1))

        lab_btns = []
        for key, text, style in self.LABELS:
            b = W.Button(description=text, button_style=style,
                         layout=W.Layout(width="110px"))
            b.on_click(self._make_label_cb(key))
            lab_btns.append(b)
        clear = W.Button(description="Clear", layout=W.Layout(width="80px"))
        clear.on_click(self._make_label_cb(None))

        self.box = W.VBox([
            self.status,
            W.HBox([prev, nxt, W.HTML("&nbsp;&nbsp;"), *lab_btns, clear]),
            W.HBox([W.VBox([W.HTML("<b>Overview</b> (context only)"), self.overview]),
                    W.VBox([W.HTML("<b>Panels</b> — 1:1 native"), self.panel_grid_box,
                            self.panel_caption, self.panel])]),
        ])

    # ---------------------------------------------------------------- callbacks
    def _make_panel_cb(self, idx):
        def cb(_):
            self.seen.setdefault(self._id(), set()).add(idx)
            _save_review(self.labels, self.seen)
            self._show_panel(idx); self._refresh_buttons()
        return cb

    def _make_label_cb(self, key):
        def cb(_):
            if key is None: self.labels.pop(self._id(), None)
            else:           self.labels[self._id()] = key
            _save_review(self.labels, self.seen)
            self._go(self.i + 1) if key else self._refresh()
        return cb

    # ---------------------------------------------------------------- internals
    def _id(self):  return self.rows[self.i]["id"]

    def _go(self, i):
        if not self.rows: return
        self.i = max(0, min(i, len(self.rows) - 1))
        self._img = None
        self._refresh()

    def _image(self):
        if self._img is None:
            self._img = Image.open(self.dir / f"{self._id()}.jpg").convert("RGB")
        return self._img

    def _show_panel(self, idx):
        im = self._image()
        n, w = self.grid, self._image().size[0] // self.grid
        r, c = divmod(idx, n)
        crop = im.crop((c*w, r*w, (c+1)*w, (r+1)*w))
        self.panel.value = _png_bytes(crop)
        km = w * 3.147 / 1000
        self.panel_caption.value = (f"panel {idx+1}/{n*n} &nbsp; {crop.size[0]}x{crop.size[1]} px "
                                    f"&nbsp; {km:.1f} x {km:.1f} km &nbsp; 1:1 at 3.15 m/px")

    def _refresh_buttons(self):
        seen = self.seen.get(self._id(), set())
        for k, b in enumerate(self.panel_btns):
            b.button_style = "info" if k in seen else ""

    def _refresh(self):
        if not self.rows:
            self.status.value = ("<b>No previews found.</b> Run stage 7's "
                                 "<code>preview_batch</code> first."); return
        r   = self.rows[self.i]
        lab = self.labels.get(r["id"], "—")
        seen, tot = len(self.seen.get(r["id"], set())), self.grid ** 2
        done = len(self.labels)
        clear = r.get("aoi_clear")
        clear = f"{clear:.3f}" if isinstance(clear, float) and clear == clear else "n/a"
        self.status.value = (
            f"<div style='font-family:monospace'>"
            f"<b>{self.i+1}/{len(self.rows)}</b> &nbsp; {r['id']} &nbsp; "
            f"{str(r.get('date','?'))} &nbsp; clear {clear} &nbsp;|&nbsp; "
            f"label: <b>{lab}</b> &nbsp; panels seen: <b>{seen}/{tot}</b> "
            f"&nbsp;|&nbsp; labelled {done}/{len(self.rows)}</div>")
        ov = self._image().copy(); ov.thumbnail((self.overview_px, self.overview_px))
        self.overview.value = _png_bytes(ov)
        self.panel.value = b""
        self.panel_caption.value = "click a panel to inspect at native resolution"
        self._refresh_buttons()

    def show(self):
        if self.missing:
            print(f"{self.missing} scene(s) in the queue have no preview file - not reviewable here")
        self._refresh(); display(self.box); return self


# Prefer the live df when the gate chain has been run this session; otherwise fall back
# to the queue stage 5 persisted. This is what makes resuming a review cheap: stage 1
# plus this cell, with no auth, no search and no rebuild.
try:
    _rows, _src = df, "live df"
except NameError:
    if not SURVIVORS_CSV.exists():
        raise RuntimeError(
            f"No df in memory and no {SURVIVORS_CSV.name}. Run cells 3-17 once to build "
            f"the review queue, then this cell alone is enough on later sittings.")
    _rows, _src = pd.read_csv(SURVIVORS_CSV), SURVIVORS_CSV.name

print(f"reviewing {len(_rows)} scenes from {_src}")
reviewer = Reviewer(_rows).show()

In [ ]:
# Read back what the reviewer recorded. Nothing is typed by hand, so a scene can only
# enter the order if you actually clicked it.
_labels, _seen = _load_review()

VESSEL_VISIBLE = sorted(i for i, v in _labels.items() if v == "vessel")
UNSURE         = sorted(i for i, v in _labels.items() if v == "unsure")

# Same source rule as the reviewer: use the live df when the gate chain has been run
# this session, otherwise restore it from the persisted queue. Without this the review
# is two cells but collecting the result still needs the whole chain rebuilt, which
# defeats the point of persisting the queue in the first place.
try:
    df
except NameError:
    if not SURVIVORS_CSV.exists():
        raise RuntimeError(
            f"No df in memory and no {SURVIVORS_CSV.name}. Run cells 3-17 once to build "
            f"the review queue.")
    df = pd.read_csv(SURVIVORS_CSV)
    if "acquired" in df.columns:                  # stages 8-9 compare real timestamps
        df["acquired"] = pd.to_datetime(df["acquired"])
    print(f"df restored from {SURVIVORS_CSV.name} ({len(df)} scenes)")

df["vessel_visible"] = df["id"].isin(VESSEL_VISIBLE)
df["unsure"]         = df["id"].isin(UNSURE)
df["reviewed"]       = df["id"].isin(_labels)
df["panels_seen"]    = df["id"].map(lambda i: len(_seen.get(i, [])))

N_PANELS = PANEL_GRID ** 2
print(f"vessel visible : {df['vessel_visible'].sum()}")
print(f"unsure         : {df['unsure'].sum()}")
print(f"no vessel      : {(df['reviewed'] & ~df['vessel_visible'] & ~df['unsure']).sum()}")
print(f"NOT reviewed   : {(~df['reviewed']).sum()}")

# A negative call is only as good as the area it was made over. These are the scenes
# where "no vessel" means "no vessel in the part of the AOI I opened" - which is not the
# same claim, and is the one that would quietly bias the visible-vs-audible table.
shallow = df[df["reviewed"] & ~df["vessel_visible"] & ~df["unsure"]
             & (df["panels_seen"] < N_PANELS)]
if len(shallow):
    print(f"\n{len(shallow)} negative call(s) made without opening all {N_PANELS} panels:")
    for _, r in shallow.sort_values("panels_seen").iterrows():
        print(f"  {r['id']}  {r['panels_seen']}/{N_PANELS} panels seen")
    print("  -> revisit these before treating them as true negatives")

df.to_csv(WORK / "screened_candidates.csv", index=False)

## 8. Acoustic join — optional

Off by default (`ACOUSTIC_JOIN = False`). It adds columns and prints a contingency table but
**gates nothing** — selection depends only on the three gates above.

If you do enable it, the visible-but-not-audible cell is the scientifically interesting one:
that is where acoustic detection range falls off, and it is why the AOI is wider than the
assumed detection radius.

In [ ]:
if ACOUSTIC_JOIN:
    det = pd.read_csv("hydrophone_detections.csv", parse_dates=["start_utc", "end_utc"])
    WINDOW = timedelta(minutes=15)
    df["n_detections"] = df["acquired"].apply(
        lambda t: int(((det["start_utc"] <= t + WINDOW) &
                       (det["end_utc"]   >= t - WINDOW)).sum()))
    df["audible"] = df["n_detections"] > 0
    print(pd.crosstab(df["vessel_visible"], df["audible"],
                      rownames=["visible"], colnames=["audible"]))
else:
    df["n_detections"], df["audible"] = 0, False
    print("Acoustic join disabled (ACOUSTIC_JOIN = False)")

## 9. Select and order

No scoring function. The three gates have already done the selection; all that remains is to
handle same-day duplicates and, if more scenes passed than the monthly budget allows, take
the cleanest first.

In [ ]:
# Option C orders on the TWO hard gates only - containment and cloud. Vessel presence
# is no longer a selection criterion: §1(2) makes empty scenes the ambient noise floor,
# so EMPTY, SINGLE-DOMINANT and CROWDED all have a defined use in §9 and none of them
# is waste. There is therefore no `vessel_visible` filter here and gate 3 need not run
# before ordering - detection happens after delivery, on NIR (§8).
try:
    sel = df.copy()
except NameError:                                  # resumed session, no kernel state
    sel = pd.read_csv(SURVIVORS_CSV)
print(f"{len(sel)} scenes passed gates 1 and 2")

if DEDUPE_BY_DAY:                                  # False - see cell 3
    before = len(sel)
    sel = (sel.sort_values("aoi_clear", ascending=False)
              .groupby("date", as_index=False).head(1))
    print(f"one-per-day dedup: {before} -> {len(sel)}")

_absent = DROP_IDS - set(sel["id"])
assert not _absent, f"DROP_IDS not present in the survivor pool: {sorted(_absent)}"
sel = sel[~sel["id"].isin(DROP_IDS)]
print(f"explicit drops ({len(DROP_IDS)}): -> {len(sel)}")

sel   = sel.sort_values("aoi_clear", ascending=False)     # cleanest first
batch = sel.head(BATCH_SIZE)

# A paired control that is not in the batch would be billed without being analysed.
_absent = set(PAIRED_VISUAL_IDS) - set(batch["id"])
assert not _absent, f"paired-visual control is not in the batch: {sorted(_absent)}"

_slots = len(batch) + len(PAIRED_VISUAL_IDS)
print(f"\nOrder: {len(batch)} analytic + {len(PAIRED_VISUAL_IDS)} visual "
      f"= {_slots} slots")
print(f"  at {CHARGE_PER_SCENE:.0f} km^2/scene : {_slots*CHARGE_PER_SCENE:.0f} of "
      f"{MONTHLY_QUOTA:.0f}  ({MONTHLY_QUOTA - _slots*CHARGE_PER_SCENE:.0f} spare)")
_dbl = (2*len(batch) + len(PAIRED_VISUAL_IDS)) * CHARGE_PER_SCENE
print(f"  if analytic bills DOUBLE   : {_dbl:.0f} of {MONTHLY_QUOTA:.0f}"
      f"  <-- OVER by {_dbl - MONTHLY_QUOTA:.0f}; this is what the probe rules out")
print(f"AOI clear range in batch: {batch['aoi_clear'].min():.3f} - "
      f"{batch['aoi_clear'].max():.3f}")

# --------------------------------------------------------------------- SPEND GATE
# Two gates, in order. PROBE_ONE spends one scene and answers the charge question;
# only then is CONFIRM_ORDER safe to set (§4).
PROBE_ONE     = False
CONFIRM_ORDER = False

In [ ]:
async def place_and_download(ids, name, bundle=BUNDLE):
    out = WORK / "downloads" / name
    out.mkdir(parents=True, exist_ok=True)
    async with Session(auth=auth) as sess:
        cl = sess.client("orders")
        req = order_request.build_request(
            name=name,
            products=[order_request.product(item_ids=ids, product_bundle=bundle,
                                            item_type=ITEM_TYPE)],
            tools=[order_request.clip_tool(aoi=aoi)],   # never skip: 6x saving
        )
        with reporting.StateBar(state="creating") as bar:
            order = await cl.create_order(req)
            bar.update(state="created", order_id=order["id"])
            await cl.wait(order["id"], callback=bar.update_state, max_attempts=0)
        await cl.download_order(order["id"], directory=out,
                                overwrite=False, progress_bar=True)
    return order["id"], out


if PROBE_ONE:
    # Settle the per-scene charge before committing the quota (§4). One scene, the
    # real bundle, the same clip tool the batch uses. Run stage 10 immediately before
    # and after: a ~100 km^2 delta means the batch above fits; ~200 means it does not
    # and BATCH_SIZE has to come down before CONFIRM_ORDER is touched.
    probe = batch.iloc[[0]]
    pid   = probe["id"].iloc[0]
    print(f"PROBE: ordering ONE scene ({pid}) as {BUNDLE}")
    oid, out = await place_and_download([pid], f"folger_probe_{pid}")
    probe.to_csv(WORK / f"manifest_probe_{pid}.csv", index=False)
    print(f"Probe order {oid} -> {out}")
    print("Re-run stage 10 and compare quota_used against the pre-probe reading.")

elif CONFIRM_ORDER:
    name = f"folger_{datetime.now():%Y%m}"
    oid, out = await place_and_download(batch["id"].tolist(), name)
    batch.to_csv(WORK / f"manifest_{name}.csv", index=False)
    print(f"Analytic order {oid} -> {out}")

    # The §5 controls: the SAME acquisitions rendered as visual. A separate order
    # because one Planet order carries one bundle.
    poid, pout = await place_and_download(PAIRED_VISUAL_IDS, f"{name}_visual",
                                          bundle=PAIRED_BUNDLE)
    print(f"Paired visual order {poid} -> {pout}")

    # NOTE: the free Data-API UDM2 re-fetch that used to live here is GONE. It existed
    # because a single-raster bundle ships no mask; analytic_sr_udm2 delivers
    # ortho_udm2 with every scene, so re-fetching would duplicate what just arrived.
    # The paired visual scenes are also in the analytic batch, so they are covered.

else:
    print("PROBE_ONE and CONFIRM_ORDER both False - nothing ordered, no quota spent.")

## 10. Check both budgets

In [ ]:
r = httpx.get("https://api.planet.com/auth/v1/experimental/public/my/subscriptions",
              auth=(API_KEY, ""), timeout=30)
for s in r.json():
    if s.get("state") == "active":
        print(f"{s.get('plan',{}).get('name')}: "
              f"{s.get('quota_used')} / {s.get('quota_sqkm')} km^2")

led = _ledger()
print(f"\nTiles ({led['month']}): {led['used']:,} / {TILE_BUDGET:,}")

## Next steps

`manifest_*.csv` carries the UTC `acquired` timestamp for each ordered scene — the join key
back to the hydrophone record.

Both quotas reset on the calendar month and neither rolls over, so run a monthly cadence
rather than saving up. If more than 30 scenes pass the gates, the surplus stays in `sel` and
you simply re-run stage 9 next month.

**Resolved — tile access.** Items on this plan carry no `_links.tiles`, and the item
`_permissions` list contains only `assets.*:download`, so the route has to be supplied by
hand. The working one is

```
https://tiles.planet.com/data/v1/PSScene/{item_id}/{z}/{x}/{y}.png
```

authenticated with the API key as HTTP basic user (or `?api_key=`). It needs **no asset
activation** — every asset on these items reports `status="inactive"` and tiles still
render. Verified across the coverage-gated pool, 2020–2025, PS2.SD and PSB.SD.

The earlier `200 / 820 bytes` result was a **fully transparent nodata tile**, not a working
route: the probe used `df.iloc[0]`, which covers only 14% of the AOI and does not contain
the hydrophone, so the requested tile fell outside its footprint. The tile service answers
200 for any well-formed request and encodes "no data here" in the alpha channel, so status
code alone cannot distinguish a broken route from an uncovered target. Stage 7 classifies
every tile as `nodata` / `flat` / `imagery` and the ledger counts only tiles actually served.

**Resolved — full-extent targeting.** Gate 1 is a topological containment test in an
equal-area projection rather than a `>= 99.9%` area ratio, and the gate 3 review box is the
AOI itself rather than an inner 5 × 5 km crop:

| | before | after |
|---|---|---|
| scenes passing gate 1 | 50 | 48 |
| AOI inspected in gate 3 | 25% | 100% |
| tiles per scene | 56 | 196 |

**Resolved — 30 scenes per month.** Stage 1 insets the nominal box by `AOI_INSET_M = 5.0` m
per edge so both plausible billing bases land under the 100 km² clip minimum — polygon
99.800 km², bounding box 99.889 km², both asserted. The pessimistic third case (bbox plus a
full 3 m pixel snap on every edge) comes to 100.009 km², marginally over; ~6 m would close
it.

**Run history.** 48 previews fetched at native resolution (226 MB, 9,236 tiles), 16 scenes
repatched after 503 losses (3,136 tiles) — 12,372 of 98,249 for the month. UDM2 screened all
48 in 27 min at concurrency 8. Gate 2: 48 → 30 survivors, `aoi_clear` mostly 1.000, with
`cloud_cover` misclassifying 7 of 48 (3 recovered, 4 rejected) against the AOI-measured
truth.

**Still worth resolving:**

1. **Settle the billing basis empirically.** Stage 10 reads `quota_used`. Order one scene,
   read the delta: 100.00 confirms the polygon basis; anything higher means raise
   `AOI_INSET_M`. Do this before committing a full 30-scene batch.
2. Confirm bare `analytic_sr` / `analytic_8b_sr` bundle names with `planet orders bundles`
   before switching off `visual`.
3. Tiles are screening only — reprojected, lossy, no radiometry. Anything entering a figure
   or a measurement comes from the ordered scene.
4. Validate the stretch against a date with a known vessel passage before trusting a
   negative. The stretch makes wakes and surface texture visible, but it has not been
   calibrated against ground truth.
5. The 30-scene batch consumes the quota exactly (30 × 100.00 = 3,000.0), leaving no headroom
   for a re-order. Plan on 29 as the number you can count on. Raising `AOI_INSET_M` does not
   buy quota headroom — the 100 km² clip minimum already dominates — it only guards the
   billing basis.
6. The year distribution is lopsided: after gate 2, 2020 has 2 scenes against 2021's 8. Any
   interannual comparison should treat 2020 as a caveat rather than a data point.